# AusculTek - Full 400 Epoch Training

**Before running:** Runtime > Change runtime type > **T4 GPU**

Then run cells in order.

In [ ]:
# 1. Setup
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q librosa torchaudio pandas

In [ ]:
# 2. Clone repo
import os

REPO_DIR = '/content/DigitalAuscultationAnalysis'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/bruhchup/DigitalAuscultationAnalysis.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# 3. Upload ICBHI dataset
# Upload your ICBHI_final_database.zip using the file browser on the left,
# or run this cell to use the upload dialog:
import glob

DATA_DIR = 'data/ICBHI_final_database'
os.makedirs(DATA_DIR, exist_ok=True)

wav_count = len(glob.glob(os.path.join(DATA_DIR, '*.wav')))

if wav_count == 0:
    from google.colab import files
    print("Upload your ICBHI_final_database.zip:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -q {zip_name} -d {DATA_DIR}/
    # Handle nested directory if zip extracts into a subfolder
    nested = os.path.join(DATA_DIR, 'ICBHI_final_database')
    if os.path.isdir(nested):
        !mv {nested}/* {DATA_DIR}/
        !rmdir {nested}

wav_count = len(glob.glob(os.path.join(DATA_DIR, '*.wav')))
txt_count = len(glob.glob(os.path.join(DATA_DIR, '*.txt')))
print(f"Dataset ready: {wav_count} wav, {txt_count} annotations")

In [ ]:
# 4. Generate metadata CSV
if not os.path.exists(os.path.join(DATA_DIR, 'metadata.csv')):
    !python -m audio_preprocessing.scl.generate_metadata --datadir {DATA_DIR}
else:
    print("metadata.csv already exists")

In [ ]:
# 5. Download PANNs pretrained weights
WEIGHTS_DIR = 'models/panns_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

weights_path = os.path.join(WEIGHTS_DIR, 'Cnn6_mAP=0.343.pth')
if not os.path.exists(weights_path):
    print("Downloading CNN6 pretrained weights...")
    !wget -q -O {weights_path} "https://zenodo.org/record/3987831/files/Cnn6_mAP%3D0.343.pth"
print(f"Weights ready: {os.path.getsize(weights_path) / 1e6:.1f} MB")

In [ ]:
# 6. TRAIN - 400 epochs hybrid SCL (~1-2 hrs on T4)
!python -m audio_preprocessing.scl.train \
    --method hybrid \
    --backbone cnn6 \
    --epochs 400 \
    --bs 128 \
    --lr 1e-4 \
    --device cuda:0 \
    --datapath {DATA_DIR} \
    --weightspath {WEIGHTS_DIR} \
    --save_dir models/scl_model \
    --tau 0.06 \
    --alpha 0.5

In [ ]:
# 7. Results
import json

with open('models/scl_model/model_results.json') as f:
    results = json.load(f)

print("="*50)
print("TRAINING RESULTS")
print("="*50)
print(f"ICBHI Score: {results['best_icbhi_score']:.4f}")
print(f"Sensitivity: {results['best_sensitivity']:.4f}")
print(f"Specificity: {results['best_specificity']:.4f}")
print(f"Epochs:      {results['epochs']}")

In [ ]:
# 8. Download trained model
!cd models && zip -r scl_model.zip scl_model/

from google.colab import files
files.download('models/scl_model.zip')

print("\nExtract scl_model.zip into your local models/ directory to replace the old model.")